# DFT, IDFT e FFT em GPU com CuPy (Google Colab)

Este notebook leva os algoritmos dos notebooks `Algoritmo_DFT_e_IDFT` e `Algoritmo_FFT` para a GPU com **CuPy** e compara o tempo de execução com as versões em CPU.

 **Antes de rodar:** no Colab, vá em *Ambiente de execução → Alterar tipo de ambiente de execução* e escolha **GPU (T4)**. O Colab dá acesso a uma GPU NVIDIA real, então os tempos medidos são de uma placa de verdade, não de uma simulação.

# **Algoritmos comparados**

| Grupo | Algoritmo | Onde roda |
|---|---|---|
| DFT | DFT com laços Python (a sua `dft_imp` original) | CPU |
| DFT | DFT matricial vetorizada (`W @ x`) | CPU (NumPy) |
| DFT | DFT matricial vetorizada (`W @ x`) | **GPU (CuPy)** |
| FFT | FFT radix-2 recursiva (a sua `fft_radix2`) | CPU |
| FFT | `np.fft.fft` | CPU |
| FFT | FFT radix-2 iterativa | **GPU (CuPy)** |
| FFT | FFT radix-2 recursiva (port direto) | **GPU (CuPy)** |
| FFT | `cp.fft.fft` (cuFFT) | **GPU (CuPy)** |

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

import sys, subprocess
try:
    import cupy as cp
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'cupy-cuda12x'])
    import cupy as cp

# %%
import time, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

props = cp.cuda.runtime.getDeviceProperties(0)
print("GPU :", props["name"].decode())
print("CuPy:", cp.__version__)

# 1. Algoritmos em CPU
As funções abaixo são cópias das suas, para o notebook ser autônomo no Colab (lá o `from ipynb.fs.full...` não existe). Incluí também a DFT matricial vetorizada do NumPy, que é o equivalente direto da versão em GPU.

--- DFT/IDFT "arcaicas": matriz W montada com laços e produto matriz-vetor com laços ---

In [ ]:
def dft_imp(x):
    N = len(x)
    W = np.zeros((N, N), dtype=complex)
    for k in range(N):
        for n in range(N):
            angulo = -2 * math.pi * k * n / N
            W[k, n] = complex(math.cos(angulo), math.sin(angulo))
    X = np.zeros(N, dtype=complex)
    for k in range(N):
        soma = complex(0.0, 0.0)
        for n in range(N):
            soma += W[k, n] * x[n]
        X[k] = soma
    return X

def idft_imp(X):
    N = len(X)
    W_inv = np.zeros((N, N), dtype=complex)
    for n in range(N):
        for k in range(N):
            angulo = 2 * math.pi * k * n / N
            W_inv[n, k] = complex(math.cos(angulo), math.sin(angulo))
    x = np.zeros(N, dtype=complex)
    for n in range(N):
        soma = complex(0.0, 0.0)
        for k in range(N):
            soma += W_inv[n, k] * X[k]
        x[n] = soma / N
    return x

# --- DFT matricial vetorizada em NumPy (sem laços Python) ---
def dft_numpy(x):
    N = len(x)
    n = np.arange(N)
    k = n.reshape((N, 1))
    W = np.exp(-1j * 2 * np.pi * k * n / N)
    return np.dot(W, x)

# --- FFT radix-2 recursiva (Cooley-Tukey DIT) ---
def fft_radix2(x):
    x = np.asarray(x, dtype=complex)
    N = len(x)
    if N <= 1:
        return x
    if N % 2 != 0:
        raise ValueError("O tamanho do sinal deve ser uma potência de 2.")
    pares = fft_radix2(x[0::2])
    impares = fft_radix2(x[1::2])
    k = np.arange(N // 2)
    twiddle = np.exp(-2j * np.pi * k / N)
    return np.concatenate([pares + twiddle * impares, pares - twiddle * impares])

def ifft_radix2(X):
    X = np.asarray(X, dtype=complex)
    return np.conj(fft_radix2(np.conj(X))) / len(X)

# 2. Algoritmos em GPU (CuPy)

 **DFT.** A matriz $W$ é construída direto na GPU, sem laços, e o produto $W\cdot x$ é feito pelo CuPy. O expoente usa $(k\cdot n) \bmod N$ em inteiros antes de entrar no `exp`, o que mantém a precisão para $N$ grande.

**FFT.** Há duas versões:

**Iterativa (a principal).** Faz a permutação *bit-reversal* e depois $\log_2 N$ estágios de borboletas, cada um vetorizado na GPU. É a mesma conta da sua FFT radix-2, só que sem recursão.
**Recursiva (port direto).** Copia a sua `fft_radix2` trocando `np` por `cp`. Está aqui para mostrar o problema: a recursão dispara milhares de *kernels* minúsculos, e a GPU perde muito tempo só em lançamentos.

In [ ]:
# ---------------- DFT / IDFT em GPU ----------------
def dft_cupy(x):
    x = cp.asarray(x, dtype=cp.complex128)
    N = x.size
    n = cp.arange(N, dtype=cp.int64)
    idx = (n[:, None] * n[None, :]) % N          # k*n mod N (inteiro)
    W = cp.exp((-2j * cp.pi / N) * idx)          # matriz de Fourier N x N
    return W @ x

def idft_cupy(X):
    X = cp.asarray(X, dtype=cp.complex128)
    N = X.size
    n = cp.arange(N, dtype=cp.int64)
    idx = (n[:, None] * n[None, :]) % N
    W_inv = cp.exp((2j * cp.pi / N) * idx)       # matriz conjugada
    return (W_inv @ X) / N


# ---------------- FFT radix-2 iterativa em GPU ----------------
def _indices_bit_reverso(N):
    bits = N.bit_length() - 1
    idx = cp.arange(N, dtype=cp.int64)
    rev = cp.zeros(N, dtype=cp.int64)
    for b in range(bits):
        rev |= ((idx >> b) & 1) << (bits - 1 - b)
    return rev

def fft_radix2_cupy(x):
    x = cp.asarray(x, dtype=cp.complex128)
    N = x.size
    if N & (N - 1) != 0:
        raise ValueError("O tamanho do sinal deve ser uma potência de 2.")
    if N <= 1:
        return x
    x = x[_indices_bit_reverso(N)]               # reordena (bit-reversal)
    m = 1                                        # metade do tamanho do bloco atual
    while m < N:
        twiddle = cp.exp(-1j * cp.pi * cp.arange(m) / m)
        x = x.reshape(-1, 2, m)                  # blocos de 2m: [metade 0 | metade 1]
        a = x[:, 0, :]
        t = twiddle * x[:, 1, :]
        x = cp.concatenate([a + t, a - t], axis=1).reshape(-1)   # borboleta
        m *= 2
    return x

def ifft_radix2_cupy(X):
    X = cp.asarray(X, dtype=cp.complex128)
    return cp.conj(fft_radix2_cupy(cp.conj(X))) / X.size


# ---------------- FFT radix-2 recursiva em GPU (port direto) ----------------
def fft_radix2_cupy_recursiva(x):
    x = cp.asarray(x, dtype=cp.complex128)
    N = x.size
    if N <= 1:
        return x
    if N % 2 != 0:
        raise ValueError("O tamanho do sinal deve ser uma potência de 2.")
    pares = fft_radix2_cupy_recursiva(x[0::2])
    impares = fft_radix2_cupy_recursiva(x[1::2])
    k = cp.arange(N // 2)
    twiddle = cp.exp(-2j * cp.pi * k / N)
    return cp.concatenate([pares + twiddle * impares, pares - twiddle * impares])


# 3. Validação

Antes de medir tempo, confere se as versões em GPU dão o mesmo resultado das de CPU (mesmos testes dos seus notebooks).


In [ ]:
# Teste 1: o sinal [1,2,1,0] do seu notebook de DFT (esperado: [4, -2j, 0, +2j])
x4 = np.array([1, 2, 1, 0], dtype=float)
print("DFT GPU [1,2,1,0]:", np.round(cp.asnumpy(dft_cupy(x4)), 4))

# Teste 2: [1..8] -- DFT em laços (CPU) x FFT manual CPU x algoritmos GPU
x8 = np.arange(1, 9, dtype=float)
ref = dft_imp(x8)
print("\nErro máximo em relação à DFT em laços (N = 8):")
print(f"  FFT manual CPU      : {np.max(np.abs(fft_radix2(x8) - ref)):.2e}")
print(f"  DFT GPU             : {np.max(np.abs(cp.asnumpy(dft_cupy(x8)) - ref)):.2e}")
print(f"  FFT GPU iterativa   : {np.max(np.abs(cp.asnumpy(fft_radix2_cupy(x8)) - ref)):.2e}")
print(f"  FFT GPU recursiva   : {np.max(np.abs(cp.asnumpy(fft_radix2_cupy_recursiva(x8)) - ref)):.2e}")
print(f"  cuFFT (cp.fft.fft)  : {np.max(np.abs(cp.asnumpy(cp.fft.fft(cp.asarray(x8))) - ref)):.2e}")

# Teste 3: N maior, contra o np.fft.fft
rng = np.random.default_rng(0)
x = rng.standard_normal(1024)
ref = np.fft.fft(x)
print("\nErro máximo em relação ao np.fft.fft (N = 1024):")
print(f"  DFT GPU             : {np.max(np.abs(cp.asnumpy(dft_cupy(x)) - ref)):.2e}")
print(f"  FFT GPU iterativa   : {np.max(np.abs(cp.asnumpy(fft_radix2_cupy(x)) - ref)):.2e}")
print(f"  FFT GPU recursiva   : {np.max(np.abs(cp.asnumpy(fft_radix2_cupy_recursiva(x)) - ref)):.2e}")

# Teste 4: ida e volta (DFT -> IDFT e FFT -> IFFT)
print("\nReconstrução x -> X -> x (erro máximo):")
print(f"  DFT/IDFT GPU        : {np.max(np.abs(cp.asnumpy(idft_cupy(dft_cupy(x))).real - x)):.2e}")
print(f"  FFT/IFFT GPU        : {np.max(np.abs(cp.asnumpy(ifft_radix2_cupy(fft_radix2_cupy(x))).real - x)):.2e}")

# 4. Configuração do benchmark

Ajuste os tamanhos e os limites abaixo se quiser. Os limites existem porque a DFT em laços e a FFT recursiva ficam lentas demais para $N$ grande. Para esses casos a tabela mostra `NaN` e o gráfico simplesmente para a curva.


In [ ]:
# %%
EXP_DFT = range(3, 14)      # DFT: N = 2^3 ... 2^13
EXP_FFT = range(3, 21)      # FFT: N = 2^3 ... 2^20
REPETICOES = 5              # número de medidas por ponto (usa a mediana)

# Limites (expoente de 2) para os algoritmos lentos
MAX_EXP_DFT_LACOS   = 10    # DFT com laços Python
MAX_EXP_DFT_NUMPY   = 12    # DFT matricial NumPy (CPU)
MAX_EXP_FFT_REC_CPU = 16    # FFT recursiva em CPU
MAX_EXP_FFT_REC_GPU = 12    # FFT recursiva em GPU

# Como o tempo é medido

A GPU trabalha de forma assíncrona: o Python recebe o controle de volta antes de a conta terminar. Por isso cada medida chama `cp.cuda.Device().synchronize()` antes de parar o relógio. Além disso:

- A primeira chamada de cada função é descartada (*warm-up*), porque inclui a inicialização do contexto CUDA e a compilação dos *kernels*.
- O valor reportado é a **mediana** de várias repetições.
- Os dados já ficam na GPU durante a medida (`sem transferência`). Há uma coluna extra, `FFT GPU c/ transferência`, que inclui copiar o sinal para a GPU e o resultado de volta, que é o custo real quando o dado nasce na CPU.


In [ ]:
def sincronizar():
    cp.cuda.Device().synchronize()

def medir(f, *args, gpu=False, repeticoes=REPETICOES):
    # Tempo (s) de f(*args): aquecimento + mediana das repetições.
    t0 = time.perf_counter()
    f(*args)
    if gpu:
        sincronizar()
    t_aquec = time.perf_counter() - t0
    if (not gpu) and t_aquec > 1.0:
        return t_aquec                      # algoritmo lento: uma medida basta
    tempos = []
    for _ in range(repeticoes):
        if gpu:
            sincronizar()
        t0 = time.perf_counter()
        f(*args)
        if gpu:
            sincronizar()
        tempos.append(time.perf_counter() - t0)
    return float(np.median(tempos))

def liberar_gpu():
    cp.get_default_memory_pool().free_all_blocks()

# 5. Benchmark da DFT

In [ ]:
rng = np.random.default_rng(42)
linhas = []

for e in EXP_DFT:
    N = 2 ** e
    x = rng.standard_normal(N)
    x_gpu = cp.asarray(x)
    ref = np.fft.fft(x)

    r = {"N": N}
    r["DFT CPU (laços)"]        = medir(dft_imp, x)    if e <= MAX_EXP_DFT_LACOS else np.nan
    r["DFT CPU (matriz NumPy)"] = medir(dft_numpy, x)  if e <= MAX_EXP_DFT_NUMPY else np.nan
    r["DFT GPU (CuPy)"]         = medir(dft_cupy, x_gpu, gpu=True)
    r["Erro DFT GPU"]           = float(np.max(np.abs(cp.asnumpy(dft_cupy(x_gpu)) - ref)))
    linhas.append(r)
    liberar_gpu()

df_dft = pd.DataFrame(linhas).set_index("N")
tempos_cols = [c for c in df_dft.columns if not c.startswith("Erro")]
tabela = df_dft.copy()
tabela[tempos_cols] = tabela[tempos_cols] * 1e3          # segundos -> milissegundos
pd.options.display.float_format = "{:.4g}".format
print("Tempos em milissegundos (ms); erro em relação ao np.fft.fft")
tabela

# %%
plt.figure(figsize=(10, 5))
estilos = {"DFT CPU (laços)": ("o-", "red"),
           "DFT CPU (matriz NumPy)": ("s-", "orange"),
           "DFT GPU (CuPy)": ("^-", "purple")}
for col, (fmt, cor) in estilos.items():
    plt.plot(df_dft.index, df_dft[col], fmt, color=cor, label=col)
plt.xscale("log", base=2); plt.yscale("log")
plt.xlabel("Tamanho do sinal ($N$)"); plt.ylabel("Tempo de execução (s)")
plt.title("DFT: CPU (laços) x CPU (NumPy) x GPU (CuPy)")
plt.grid(True, which="both", ls="--", alpha=0.6); plt.legend()
plt.tight_layout(); plt.show()


# 6. Benchmark da FFT


In [ ]:
def fft_gpu_com_transferencia(x):
    return cp.asnumpy(fft_radix2_cupy(cp.asarray(x)))

rng = np.random.default_rng(42)
linhas = []

for e in EXP_FFT:
    N = 2 ** e
    x = rng.standard_normal(N)
    x_gpu = cp.asarray(x)
    ref = np.fft.fft(x)

    r = {"N": N}
    r["FFT manual CPU (recursiva)"] = medir(fft_radix2, x) if e <= MAX_EXP_FFT_REC_CPU else np.nan
    r["np.fft.fft (CPU)"]           = medir(np.fft.fft, x)
    r["FFT manual GPU (iterativa)"] = medir(fft_radix2_cupy, x_gpu, gpu=True)
    r["FFT manual GPU (recursiva)"] = (medir(fft_radix2_cupy_recursiva, x_gpu, gpu=True)
                                       if e <= MAX_EXP_FFT_REC_GPU else np.nan)
    r["cuFFT (cp.fft.fft)"]         = medir(cp.fft.fft, x_gpu, gpu=True)
    r["FFT GPU c/ transferência"]   = medir(fft_gpu_com_transferencia, x, gpu=True)
    r["Erro FFT GPU iterativa"]     = float(np.max(np.abs(cp.asnumpy(fft_radix2_cupy(x_gpu)) - ref)))
    linhas.append(r)
    liberar_gpu()

df_fft = pd.DataFrame(linhas).set_index("N")
tempos_cols = [c for c in df_fft.columns if not c.startswith("Erro")]
tabela = df_fft.copy()
tabela[tempos_cols] = tabela[tempos_cols] * 1e3
print("Tempos em milissegundos (ms); erro em relação ao np.fft.fft")
tabela

# %%
plt.figure(figsize=(11, 6))
estilos = {"FFT manual CPU (recursiva)": ("s-", "blue"),
           "np.fft.fft (CPU)": ("^--", "green"),
           "FFT manual GPU (iterativa)": ("o-", "purple"),
           "FFT manual GPU (recursiva)": ("x:", "gray"),
           "cuFFT (cp.fft.fft)": ("D-", "red"),
           "FFT GPU c/ transferência": ("v--", "magenta")}
for col, (fmt, cor) in estilos.items():
    plt.plot(df_fft.index, df_fft[col], fmt, color=cor, label=col)
plt.xscale("log", base=2); plt.yscale("log")
plt.xlabel("Tamanho do sinal ($N$)"); plt.ylabel("Tempo de execução (s)")
plt.title("FFT: CPU x GPU")
plt.grid(True, which="both", ls="--", alpha=0.6); plt.legend()
plt.tight_layout(); plt.show()

# Aceleração em relação ao np.fft.fft (valores > 1 significam que a GPU é mais rápida)
acel = pd.DataFrame({
    "FFT manual GPU / np.fft": df_fft["np.fft.fft (CPU)"] / df_fft["FFT manual GPU (iterativa)"],
    "cuFFT / np.fft":          df_fft["np.fft.fft (CPU)"] / df_fft["cuFFT (cp.fft.fft)"],
    "GPU c/ transferência / np.fft": df_fft["np.fft.fft (CPU)"] / df_fft["FFT GPU c/ transferência"],
})
plt.figure(figsize=(10, 4.5))
for col in acel.columns:
    plt.plot(acel.index, acel[col], "o-", label=col)
plt.axhline(1.0, color="k", ls="--", lw=1)
plt.xscale("log", base=2); plt.yscale("log")
plt.xlabel("Tamanho do sinal ($N$)"); plt.ylabel("Aceleração (x vezes)")
plt.title("Aceleração da GPU em relação ao np.fft.fft")
plt.grid(True, which="both", ls="--", alpha=0.6); plt.legend()
plt.tight_layout(); plt.show()
acel

# 7. Bônus: convolução linear via FFT em GPU

Mesma ideia do seu notebook de convolução (zero-padding até $L = N_1 + N_2 - 1$, arredondado para a próxima potência de 2), agora com a FFT em GPU. O sinal $x$ é longo e o filtro $h$ tem 1025 coeficientes.


In [ ]:
def conv_linear_fft(x, h, xp, fft, ifft):
    # Convolução linear via FFT. xp = np ou cp; fft/ifft = funções de transformada.
    L = len(x) + len(h) - 1
    N = 1 << (L - 1).bit_length()                 # próxima potência de 2 >= L
    X = fft(xp.pad(x, (0, N - len(x))))
    H = fft(xp.pad(h, (0, N - len(h))))
    return ifft(X * H).real[:L]

# Validação com a amostra dos seus notebooks
x = np.array([3, 1, 4, 1, 5, 9, 2, 6], dtype=float)
h = np.array([1, 2, 1], dtype=float)
y_gpu = cp.asnumpy(conv_linear_fft(cp.asarray(x), cp.asarray(h), cp, fft_radix2_cupy, ifft_radix2_cupy))
print("GPU     :", np.round(y_gpu, 4))
print("np.conv.:", np.convolve(x, h))

# %%
EXP_CONV = range(12, 21)          # |x| = 2^12 ... 2^20
M = 1025                          # coeficientes do filtro
MAX_EXP_CONV_FFT_CPU_MANUAL = 15  # FFT manual recursiva em CPU

rng = np.random.default_rng(7)
h = rng.standard_normal(M)
h_gpu = cp.asarray(h)
linhas = []

for e in EXP_CONV:
    x = rng.standard_normal(2 ** e)
    x_gpu = cp.asarray(x)
    ref = np.convolve(x, h)

    def conv_gpu_com_transf(x, h):
        return cp.asnumpy(conv_linear_fft(cp.asarray(x), cp.asarray(h), cp,
                                          fft_radix2_cupy, ifft_radix2_cupy))

    r = {"|x|": 2 ** e}
    r["np.convolve (direta)"]       = medir(np.convolve, x, h)
    r["FFT manual CPU"]             = (medir(conv_linear_fft, x, h, np, fft_radix2, ifft_radix2)
                                       if e <= MAX_EXP_CONV_FFT_CPU_MANUAL else np.nan)
    r["np.fft (CPU)"]               = medir(conv_linear_fft, x, h, np, np.fft.fft, np.fft.ifft)
    r["FFT manual GPU"]             = medir(conv_linear_fft, x_gpu, h_gpu, cp,
                                            fft_radix2_cupy, ifft_radix2_cupy, gpu=True)
    r["cuFFT (GPU)"]                = medir(conv_linear_fft, x_gpu, h_gpu, cp,
                                            cp.fft.fft, cp.fft.ifft, gpu=True)
    r["FFT manual GPU c/ transf."]  = medir(conv_gpu_com_transf, x, h, gpu=True)
    r["Erro (FFT manual GPU)"]      = float(np.max(np.abs(
        cp.asnumpy(conv_linear_fft(x_gpu, h_gpu, cp, fft_radix2_cupy, ifft_radix2_cupy)) - ref)))
    linhas.append(r)
    liberar_gpu()

df_conv = pd.DataFrame(linhas).set_index("|x|")
tempos_cols = [c for c in df_conv.columns if not c.startswith("Erro")]
tabela = df_conv.copy()
tabela[tempos_cols] = tabela[tempos_cols] * 1e3
print(f"Convolução linear com filtro de {M} coeficientes. Tempos em ms.")
tabela

# %%
plt.figure(figsize=(11, 6))
for col in [c for c in df_conv.columns if not c.startswith("Erro")]:
    plt.plot(df_conv.index, df_conv[col], "o-", label=col)
plt.xscale("log", base=2); plt.yscale("log")
plt.xlabel("Tamanho de $x$"); plt.ylabel("Tempo de execução (s)")
plt.title(f"Convolução linear (filtro de {M} coeficientes): CPU x GPU")
plt.grid(True, which="both", ls="--", alpha=0.6); plt.legend()
plt.tight_layout(); plt.show()

# 8. Como interpretar os resultados
#
 - **N pequeno:** é comum a CPU ganhar. Cada operação na GPU tem um custo fixo de lançamento de *kernel* (da ordem de microssegundos) que domina quando o vetor é pequeno.
 - **N grande:** a GPU tende a ganhar porque os $N/2$ butterflies de cada estágio rodam em paralelo. O ponto de cruzamento depende da GPU sorteada no Colab.
 - **Transferência CPU ↔ GPU:** a coluna `c/ transferência` mostra o custo real quando o sinal nasce na CPU. Para a FFT sozinha, a cópia pela PCIe pode anular o ganho. A GPU compensa mais quando vários passos ficam na GPU (como na convolução, com duas FFTs e uma IFFT) ou quando o sinal já está lá.
 - **FFT recursiva na GPU:** deve ficar bem atrás da iterativa, por causa dos milhares de lançamentos de *kernels* pequenos. É um bom argumento para o relatório: o mesmo algoritmo precisa ser reescrito para a arquitetura.
 - **cuFFT** é a biblioteca otimizada da NVIDIA, o equivalente GPU de `np.fft.fft`. Serve de teto de desempenho para comparar com a sua FFT manual.
 - **Precisão:** tudo está em `complex128` para bater com o NumPy. Em placas como a T4, o ponto flutuante de 64 bits é bem mais lento que o de 32 bits. Se quiser testar `complex64` (erro da ordem de $10^{-6}$), troque `cp.complex128` por `cp.complex64` nas funções.
- **Medidas variam:** o Colab compartilha recursos, então rode mais de uma vez e use o mesmo tipo de GPU nas comparações.